In [ ]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
import sqlite3
import csv

connection = sqlite3.connect("db.sqlite3")
cursor = connection.cursor()
cursor.execute("""
    SELECT
        oi.id          AS order_item_id,
        oi.order_id,
        oi.product_id,
        p.name         AS product_name,
        p.price        AS product_price,
        oi.quantity,
        o.datetime     AS order_datetime
    FROM restaurant_orderitem oi
        JOIN restaurant_product p ON oi.product_id = p.id
        JOIN restaurant_order  o ON oi.order_id = o.id
    ORDER BY oi.id
""")
rows = cursor.fetchall()
columns = [description[0] for description in cursor.description]
connection.close()

with open("data.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(columns)
    writer.writerows(rows)

df = pd.read_csv("data.csv")

print(df.shape)
df.head()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_by_quantity = (
    df.groupby("product_name")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_by_quantity.plot(
    kind="pie",
    autopct="%1.1f%%",
    startangle=90,
    figsize=(8, 8),
    title="Top 10 most popular products (by quantity)",
)
plt.ylabel("")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["item_price"] = df["product_price"] * df["quantity"]

top_by_income = (
    df.groupby("product_name")["item_price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_by_income.plot(
    kind="pie",
    autopct="%1.1f%%",
    startangle=90,
    figsize=(8, 8),
    title="Top 10 products by revenue (Item Price)",
)
plt.ylabel("")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["order_datetime"] = pd.to_datetime(df["order_datetime"])
df["order_hour"] = df["order_datetime"].dt.hour

hourly_income = df.groupby("order_hour")["item_price"].sum()

hourly_income.plot(
    kind="bar",
    figsize=(12, 6),
    color="steelblue",
    title="Total restaurant income by order hour",
    xlabel="Order Hour",
    ylabel="Total income",
)
plt.xticks(rotation=0)
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["order_day"] = df["order_datetime"].dt.day_name()

daily_income = (
    df.groupby("order_day")["item_price"]
    .sum()
    .reindex(day_order)
)

daily_income.plot(
    kind="bar",
    figsize=(10, 6),
    color="seagreen",
    title="Total restaurant income by day of the week",
    xlabel="Day of the week",
    ylabel="Total income",
)
plt.xticks(rotation=0)
plt.show()